# 09 · Análisis temporal: trayectorias, interacciones, acciones y datos estructurados 

La oferta: *"análisis de trayectorias, movimientos e interacciones"*, *"procesamiento temporal de secuencias de
vídeo"*, *"transformación de resultados visuales en datos estructurados"*, *"generación de resultados
estructurados para su incorporación a una plataforma"* y valoran *"analizar acciones o patrones temporales"*.

1. Trayectorias ruidosas → suavizado ✍️ → velocidad, distancia, sprints
2. Interacciones: proximidad entre jugadores, eventos con histéresis ✍️
3. Reconocimiento de acciones: enfoques y un clasificador temporal pequeño ✍️
4. Datos estructurados: esquema, formatos y buenas prácticas

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.signal import savgol_filter

FPS = 25
rng = np.random.default_rng(0)

## 1. Trayectorias en metros

Simulamos 10 jugadores durante 60 s a 25 FPS sobre el campo, con fases de parado / trote / sprint, y las
"observamos" como saldrían del pipeline (detector → tracker → homografía): **ruido de ~0.3 m** por frame y
**huecos** (oclusiones no recuperadas).

In [ ]:
def simulate_player(T=60 * FPS):
    pos = np.zeros((T, 2)); pos[0] = rng.uniform([10, 5], [95, 63])
    speed_states = {"parado": 0.3, "andar": 1.5, "trote": 3.5, "sprint": 7.5}   # m/s
    state, heading, states = "andar", rng.uniform(0, 2 * np.pi), []
    for t in range(1, T):
        if rng.random() < 1 / (3 * FPS):                       # cambia de fase cada ~3 s
            state = rng.choice(list(speed_states), p=[0.2, 0.35, 0.3, 0.15])
        heading += rng.normal(0, 0.08)
        v = speed_states[state] * (1 + 0.1 * rng.normal())
        pos[t] = pos[t - 1] + v / FPS * np.array([np.cos(heading), np.sin(heading)])
        for d, hi in [(0, 105), (1, 68)]:                      # rebote en las bandas
            if not 0 < pos[t, d] < hi:
                heading = np.pi - heading if d == 0 else -heading
                pos[t, d] = np.clip(pos[t, d], 0.1, hi - 0.1)
        states.append(state)
    return pos, ["andar"] + states

players = {pid: simulate_player() for pid in range(1, 11)}
true_xy = {pid: p for pid, (p, _) in players.items()}
obs = {}
for pid, p in true_xy.items():
    o = p + rng.normal(0, 0.3, p.shape)
    for _ in range(4):                                         # 4 huecos de 0.5–2 s
        s = rng.integers(0, len(o) - 50); o[s:s + rng.integers(12, 50)] = np.nan
    obs[pid] = o

In [ ]:
p = 1
t = np.arange(len(obs[p])) / FPS
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
ax[0].plot(*true_xy[p].T, "k-", lw=2, label="real"); ax[0].plot(*obs[p].T, ".", ms=2, alpha=0.4, label="observada")
ax[0].set_aspect("equal"); ax[0].legend(); ax[0].set_title("trayectoria en el campo (m)")
v_raw = np.linalg.norm(np.diff(obs[p], axis=0), axis=1) * FPS
v_true = np.linalg.norm(np.diff(true_xy[p], axis=0), axis=1) * FPS
ax[1].plot(t[1:], v_raw * 3.6, alpha=0.5, label="derivada directa"); ax[1].plot(t[1:], v_true * 3.6, "k", label="real")
ax[1].set_ylabel("km/h"); ax[1].legend(); ax[1].set_title("¡derivar amplifica el ruido!"); plt.show()
print(f"distancia real {np.nansum(v_true) / FPS:.0f} m  vs  sumando la trayectoria ruidosa {np.nansum(v_raw) / FPS:.0f} m")

Con 0.3 m de ruido por frame a 25 FPS, la "velocidad" ruidosa es ~0.3·√2·25 ≈ 10 m/s (¡36 km/h!) de puro ruido,
y la distancia recorrida sale **inflada** muchísimo. Hay que **suavizar antes de derivar**.

Opciones de suavizado:
- **Media móvil**: simple, pero retrasa y aplana picos.
- **Savitzky–Golay**: ajusta un polinomio local en una ventana → suaviza conservando la forma de los picos, y da
  la **derivada analítica** (`deriv=1`). Muy usado para velocidades. Es *offline* (usa el futuro, ventana centrada).
- **Kalman** (online) / **suavizador RTS** (Kalman hacia delante + hacia atrás, offline).
- **One Euro filter**: paso-bajo con frecuencia de corte adaptada a la velocidad (online, poco retardo).

✍️ **EJERCICIO**: implementa `kinematics(xy, fps, window)`:
1. Interpolar huecos (NaN) linealmente (solo los cortos; aquí todos).
2. Savitzky–Golay (orden 2) para posición y para velocidad (`deriv=1`, multiplicar por fps).
3. Devuelve un DataFrame con `x, y, vx, vy, speed_kmh`.

In [ ]:
# SOLUCIÓN
def kinematics(xy, fps=FPS, window=25):
    df = pd.DataFrame(xy, columns=["x", "y"]).interpolate(limit_direction="both")
    out = pd.DataFrame({"t": np.arange(len(df)) / fps})
    for c in ["x", "y"]:
        out[c] = savgol_filter(df[c].values, window, 2)
        out["v" + c] = savgol_filter(df[c].values, window, 2, deriv=1) * fps
    out["speed_kmh"] = np.hypot(out.vx, out.vy) * 3.6
    return out

In [ ]:
k = kinematics(obs[p])
plt.figure(figsize=(16, 4))
plt.plot(t[1:], v_raw * 3.6, alpha=0.2, label="derivada directa")
plt.plot(k.t, k.speed_kmh, label="Savitzky–Golay")
plt.plot(t[1:], v_true * 3.6, "k", lw=1, label="real"); plt.ylabel("km/h"); plt.legend(); plt.show()
err = np.abs(k.speed_kmh.values[1:] - v_true * 3.6).mean()
print(f"error medio de velocidad: {err:.2f} km/h | distancia estimada {k.speed_kmh.sum() / 3.6 / FPS:.0f} m vs real {v_true.sum() / FPS:.0f} m")

In [ ]:
# Ajuste de la ventana: compromiso ruido / retardo-suavizado de picos
for w in [5, 11, 25, 51, 101]:
    kk = kinematics(obs[p], window=w)
    print(f"ventana {w:3d} frames ({w / FPS:.1f}s): error {np.abs(kk.speed_kmh.values[1:] - v_true * 3.6).mean():5.2f} km/h")

### Métricas físicas por jugador (lo que vendería una plataforma de analítica)

In [ ]:
def player_summary(k, sprint_kmh=24, min_sprint_s=1.0):
    fast = (k.speed_kmh > sprint_kmh).values
    # nº de sprints = tramos contiguos por encima del umbral con duración mínima
    edges = np.diff(np.r_[0, fast.astype(int), 0])
    starts, ends = np.where(edges == 1)[0], np.where(edges == -1)[0]
    n_sprints = int(((ends - starts) / FPS >= min_sprint_s).sum())
    return {"distancia_m": round(k.speed_kmh.sum() / 3.6 / FPS, 1), "vel_max_kmh": round(k.speed_kmh.max(), 1),
            "vel_media_kmh": round(k.speed_kmh.mean(), 1), "n_sprints": n_sprints,
            "dist_alta_intensidad_m": round(k.speed_kmh[k.speed_kmh > 19.8].sum() / 3.6 / FPS, 1)}

K = {pid: kinematics(o) for pid, o in obs.items()}
summary = pd.DataFrame({pid: player_summary(k) for pid, k in K.items()}).T
summary.index.name = "player_id"
summary

In [ ]:
# Mapa de calor de posiciones de un jugador
k = K[1]
plt.figure(figsize=(9, 6))
plt.hist2d(k.x, k.y, bins=[21, 14], range=[[0, 105], [0, 68]], cmap="hot"); plt.gca().invert_yaxis()
plt.colorbar(label="frames"); plt.title("mapa de calor jugador 1");

## 2. Interacciones

Con todas las trayectorias en el mismo sistema de coordenadas: **matriz de distancias entre jugadores por
frame** `(T, N, N)` → proximidad, marcajes, duelos, presión, compañero más cercano, líneas de pase, posesión
(jugador más cercano al balón), etc.

Detectar **eventos** a partir de señales temporales ruidosas: si usas un umbral simple, la señal "parpadea"
alrededor del umbral y genera decenas de eventos falsos. Solución: **histéresis** (umbral de entrada < umbral
de salida) + **duración mínima**.

✍️ **EJERCICIO**: `detect_events(signal, enter, exit, min_len)` → lista de `(inicio, fin)` donde la señal baja de
`enter` y no vuelve a superar `exit`; descarta eventos más cortos que `min_len` frames.

In [ ]:
# SOLUCIÓN
def detect_events(signal, enter, exit, min_len):
    events, active, start = [], False, 0
    for i, s in enumerate(signal):
        if not active and s < enter:
            active, start = True, i
        elif active and s > exit:
            active = False
            if i - start >= min_len:
                events.append((start, i))
    if active and len(signal) - start >= min_len:
        events.append((start, len(signal)))
    return events

In [ ]:
ids = sorted(K)
XY = np.stack([K[i][["x", "y"]].values for i in ids], axis=1)        # (T, N, 2)
D = np.linalg.norm(XY[:, :, None] - XY[:, None, :], axis=-1)           # (T, N, N)
rows = []
for a in range(len(ids)):
    for b in range(a + 1, len(ids)):
        for s, e in detect_events(D[:, a, b], enter=2.0, exit=3.0, min_len=FPS // 2):
            rows.append({"tipo": "proximidad", "jugadores": [ids[a], ids[b]], "t_inicio": s / FPS, "t_fin": e / FPS,
                         "dist_min_m": round(D[s:e, a, b].min(), 2)})
events = pd.DataFrame(rows)
naive = sum(int(np.sum(np.diff((D[:, a, b] < 2).astype(int)) == 1)) for a in range(len(ids)) for b in range(a + 1, len(ids)))
print(f"eventos con histéresis: {len(events)} | con umbral simple (sin histéresis ni duración) sobre la señal ruidosa: {naive}")
events.head()

## 3. Reconocimiento de acciones y patrones temporales

Enfoques, de más simple a más complejo:
| Enfoque | Entrada | Modelos | Cuándo |
|---|---|---|---|
| **Reglas** | trayectorias, velocidades, distancias | umbrales + histéresis | eventos bien definidos (sprint, proximidad, entrar en zona) |
| **Secuencias de features** | por frame: posición, velocidad, pose, embedding | 1D-CNN / GRU / LSTM / Transformer temporal | patrones de movimiento, acciones simples |
| **Basado en esqueleto** | secuencias de keypoints (T × 17 × 2) | **ST-GCN**, PoseC3D, MotionBERT | acciones corporales (tiro, salto, caída), robusto a fondo/ropa |
| **Basado en vídeo** | clips de píxeles (T × H × W × 3) | 3D-CNN (I3D, X3D, **SlowFast**), transformers de vídeo (TimeSformer, **VideoMAE**, ViViT) | acciones con contexto visual; caros |
| **Temporal action localization / spotting** | vídeo largo | ventanas deslizantes + NMS temporal, **SoccerNet action spotting** | encontrar *cuándo* ocurre algo (gol, falta, pase) |

Trucos prácticos: ventanas deslizantes con solapamiento, normalizar respecto a la persona (centrar en la cadera,
escalar por altura), aumentar con *time-warping*, y **NMS temporal** para no disparar el mismo evento en ventanas
consecutivas.

✍️ **EJERCICIO**: clasificador temporal pequeño (1D-CNN) que, a partir de ventanas de 2 s de velocidad
(vx, vy) de un jugador, predice su fase de movimiento (parado / andar / trote / sprint). Usamos las etiquetas
de la simulación. Ya está escrito: léelo, ejecútalo y prueba a sustituir la 1D-CNN por una `nn.GRU`.

In [ ]:
LABELS = ["parado", "andar", "trote", "sprint"]
WIN = 2 * FPS
Xs, ys = [], []
for pid, (_, states) in players.items():
    k = K[pid]; v = k[["vx", "vy"]].values.astype(np.float32)
    lab = np.array([LABELS.index(s) for s in states])
    for s in range(0, len(v) - WIN, 10):
        y = np.bincount(lab[s:s + WIN], minlength=4).argmax()         # etiqueta mayoritaria de la ventana
        Xs.append(v[s:s + WIN].T); ys.append(y)
X = torch.tensor(np.array(Xs)); Y = torch.tensor(ys)
split = int(0.8 * len(X))       # ventanas ordenadas por jugador → test = jugadores 9 y 10. ¡No aleatorio: las ventanas solapan!
Xtr, Ytr, Xte, Yte = X[:split], Y[:split], X[split:], Y[split:]
print(X.shape, np.bincount(ys, minlength=4))

model = nn.Sequential(
    nn.Conv1d(2, 32, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
    nn.Conv1d(32, 64, 5, padding=2), nn.ReLU(), nn.AdaptiveAvgPool1d(1), nn.Flatten(),
    nn.Linear(64, 4))
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
for ep in range(30):
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 64):
        b = perm[i:i + 64]
        loss = nn.functional.cross_entropy(model(Xtr[b]), Ytr[b])
        opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    pred = model(Xte).argmax(1)
acc = (pred == Yte).float().mean().item()
cm = pd.crosstab(pd.Series(Yte.numpy(), name="real").map(dict(enumerate(LABELS))),
                 pd.Series(pred.numpy(), name="pred").map(dict(enumerate(LABELS))))
print(f"accuracy test: {acc:.3f}")
cm

## 4. Datos estructurados para una plataforma

El producto final no es un vídeo con cajas, son **datos**. Buenas prácticas que conviene mencionar:
- **Tres niveles**: (1) *tracking data* por frame y objeto (tabla larga), (2) *resúmenes* por objeto/periodo,
  (3) *eventos* (con inicio, fin, actores y confianza).
- **Esquema explícito y versionado** (JSON Schema / Pydantic / Protobuf): unidades (m, m/s), sistema de
  coordenadas (origen, ejes, lado del campo), `fps` y **timestamps** absolutos, versión del modelo y del pipeline.
- **Confianza e incertidumbre**: score de detección, si la posición fue **interpolada**, calidad de la homografía.
- Formatos: **Parquet** para tablas grandes (columnar, comprimido, tipado) → frames×jugadores de un partido son
  ~3M filas; JSON para eventos/resúmenes y APIs; streaming (Kafka/WebSocket) para tiempo real.
- IDs estables: `track_id` (interno del tracker) ≠ `player_id` (identidad del negocio, tras ReID/dorsal).

In [ ]:
tracking = pd.concat([K[i].assign(track_id=i, frame=np.arange(len(K[i])),
                                  interpolated=np.isnan(obs[i][:, 0])) for i in ids])
tracking = tracking[["frame", "t", "track_id", "x", "y", "vx", "vy", "speed_kmh", "interpolated"]].round(3)
tracking.to_csv("tracking.csv", index=False)
try:
    tracking.to_parquet("tracking.parquet", index=False)
    import os; print(f"CSV {os.path.getsize('tracking.csv') / 1e6:.1f} MB vs Parquet {os.path.getsize('tracking.parquet') / 1e6:.1f} MB")
except ImportError:
    print("instala pyarrow para Parquet: pip install pyarrow")

doc = {
    "schema_version": "1.0",
    "match_id": "demo-001", "fps": FPS, "coordinate_system": {"origin": "esquina inferior izquierda", "units": "m",
                                                               "x": [0, 105], "y": [0, 68]},
    "pipeline": {"detector": "yolo11n", "tracker": "bytetrack", "homography": "manual-12pts-ransac"},
    "players": summary.reset_index().to_dict(orient="records"),
    "events": events.head(5).to_dict(orient="records"),
}
print(json.dumps(doc, indent=2, ensure_ascii=False, default=float)[:1200])